# India Bank Credit Growth — Analysis Notebook
**Source:** RBI DBIE
**Datasets:** Sectoral Deployment of Non-Food Gross Bank Credit + SCB Business in India

Run all cells top-to-bottom. Outputs saved to `data/processed/`.

In [3]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

RAW = Path('data/raw')
OUT = Path('data/processed')
OUT.mkdir(exist_ok=True)
print('✅ Ready')

✅ Ready


## 1. Sectoral Credit (Annual)

In [4]:
df1 = pd.read_csv(RAW / 'Sectoral Deployment of Non-Food Gross Bank Credit.csv', header=None)

dates = [d for d in df1.iloc[6, 4:].tolist() if str(d) != 'nan']
dates = pd.to_datetime(dates).strftime('%Y').tolist()

sectors = {
    7:  'Total Non-Food Credit',
    9:  'Agriculture',
    10: 'Industry',
    12: 'Micro & Small',
    13: 'Medium',
    14: 'Large',
    15: 'Services',
    33: 'Personal Loans',
    23: 'Trade',
    26: 'Commercial Real Estate',
    28: 'NBFCs',
}

sectoral_data = []
for row_idx, sector_name in sectors.items():
    values = []
    for col in range(4, 4+len(dates)):
        v = df1.iloc[row_idx, col]
        try:
            v = str(v).replace('(','').replace(')','').replace(',','')
            values.append(round(float(v)/100000, 2))
        except:
            values.append(None)
    sectoral_data.append({'sector': sector_name, 'values': values, 'years': dates})

print(f'✅ Sectoral data: {len(sectoral_data)} sectors × {len(dates)} years')
print('Years:', dates)
pd.DataFrame([{'sector': s['sector'], **{y: v for y, v in zip(dates, s['values'])}} for s in sectoral_data])

✅ Sectoral data: 11 sectors × 7 years
Years: ['2019', '2020', '2021', '2022', '2023', '2024', '2025']


,sector,2019,2020,2021,2022,2023,2024,2025
0,Total Non-Food Credit,97.30,103.19,108.88,118.36,136.55,164.09,183.72
1,Agriculture,11.13,12.03,13.30,14.96,17.26,20.71,22.87
2,Industry,28.38,29.47,29.41,31.81,33.66,36.82,39.86
3,Micro & Small,3.71,4.03,4.33,5.60,6.33,7.33,7.98
4,Medium,1.01,1.09,1.45,2.39,2.68,3.06,3.63
5,Large,23.65,24.35,23.62,23.82,24.65,26.43,28.24
6,Services,23.41,26.72,27.65,31.13,37.19,45.47,50.94
7,Personal Loans,23.03,27.27,30.09,34.66,41.83,53.47,59.72
8,Trade,4.80,5.57,6.28,7.36,8.72,10.24,11.85
9,Commercial Real Estate,2.28,2.88,2.89,2.97,3.23,4.60,5.23


## 2. YoY Sectoral Growth

In [5]:
sectoral_growth = []
for item in sectoral_data:
    growth = [None]
    for i in range(1, len(item['values'])):
        if item['values'][i] and item['values'][i-1]:
            g = round((item['values'][i] - item['values'][i-1]) / item['values'][i-1] * 100, 2)
            growth.append(g)
        else:
            growth.append(None)
    sectoral_growth.append({'sector': item['sector'], 'growth': growth, 'years': dates})

print('✅ Growth rates computed')
pd.DataFrame([{'sector': s['sector'], **{y: g for y, g in zip(dates, s['growth'])}} for s in sectoral_growth])

✅ Growth rates computed


,sector,2019,2020,2021,2022,2023,2024,2025
0,Total Non-Food Credit,None,6.05,5.51,8.71,15.37,20.17,11.96
1,Agriculture,None,8.09,10.56,12.48,15.37,19.99,10.43
2,Industry,None,3.84,-0.20,8.16,5.82,9.39,8.26
3,Micro & Small,None,8.63,7.44,29.33,13.04,15.80,8.87
4,Medium,None,7.92,33.03,64.83,12.13,14.18,18.63
5,Large,None,2.96,-3.00,0.85,3.48,7.22,6.85
6,Services,None,14.14,3.48,12.59,19.47,22.26,12.03
7,Personal Loans,None,18.41,10.34,15.19,20.69,27.83,11.69
8,Trade,None,16.04,12.75,17.20,18.48,17.43,15.72
9,Commercial Real Estate,None,26.32,0.35,2.77,8.75,42.41,13.70


## 3. Monthly Bank Credit

In [6]:
df2 = pd.read_csv(RAW / 'WSS Table No. 04 _ Scheduled Commercial Banks - Business in India_SBDMS.csv', header=None)

monthly_rows = []
for i in range(6, len(df2)):
    date_val = str(df2.iloc[i, 1]).strip()
    if date_val in ('nan', '', 'Fortnight Date Final', 'See Notes on Tables.'):
        continue
    try:
        date = pd.to_datetime(date_val, errors='coerce')
        if pd.isna(date) or date.year < 2010:
            continue
        monthly_rows.append({
            'date': date.strftime('%Y-%m-%d'),
            'bank_credit': round(float(str(df2.iloc[i, 24]).replace(',',''))/100000, 2),
            'deposits': round(float(str(df2.iloc[i, 7]).replace(',',''))/100000, 2),
            'nonfood_credit': round(float(str(df2.iloc[i, 26]).replace(',',''))/100000, 2),
        })
    except:
        continue

monthly_rows = sorted(monthly_rows, key=lambda x: x['date'])

# YoY growth (26 fortnights ≈ 1 year)
for i in range(len(monthly_rows)):
    if i >= 26:
        prev = monthly_rows[i-26]['bank_credit']
        curr = monthly_rows[i]['bank_credit']
        monthly_rows[i]['credit_growth_yoy'] = round((curr-prev)/prev*100, 2) if prev else None
    else:
        monthly_rows[i]['credit_growth_yoy'] = None

print(f'✅ Monthly data: {len(monthly_rows)} fortnights')
print(f'Range: {monthly_rows[0]["date"]} → {monthly_rows[-1]["date"]}')
pd.DataFrame(monthly_rows).tail()

✅ Monthly data: 426 fortnights
Range: 2010-01-01 → 2026-04-30


,date,bank_credit,deposits,nonfood_credit,credit_growth_yoy
421,2026-02-28,207.51,251.90,205.86,15.35
422,2026-03-15,207.70,250.11,206.95,14.59
423,2026-03-31,213.60,262.30,212.90,17.08
424,2026-04-15,209.18,256.48,208.49,13.67
425,2026-04-30,212.12,258.64,211.08,16.63


## 4. Sector Shares & Summary

In [7]:
latest_year_data = {s['sector']: s['values'][-1] for s in sectoral_data if s['values'][-1]}
total = latest_year_data.get('Total Non-Food Credit', 1)
sector_shares = [
    {'sector': k, 'value': v, 'share': round(v/total*100, 2)}
    for k, v in latest_year_data.items() if k != 'Total Non-Food Credit'
]

latest = monthly_rows[-1]
summary = {
    'latest_date': latest['date'],
    'total_credit_lakh_cr': latest['bank_credit'],
    'total_deposits_lakh_cr': latest['deposits'],
    'credit_growth_yoy': latest.get('credit_growth_yoy'),
    'cd_ratio': round(latest['bank_credit']/latest['deposits']*100, 2),
    'latest_sectoral_year': dates[-1],
    'total_nonfood_credit_lakh_cr': sectoral_data[0]['values'][-1],
}

print('Summary:', summary)

Summary: {'latest_date': '2026-04-30', 'total_credit_lakh_cr': 212.12, 'total_deposits_lakh_cr': 258.64, 'credit_growth_yoy': 16.63, 'cd_ratio': 82.01, 'latest_sectoral_year': '2025', 'total_nonfood_credit_lakh_cr': 183.72}


## 5. Export JSON

In [8]:
files = {
    'credit_monthly.json': monthly_rows,
    'credit_sectoral.json': sectoral_data,
    'credit_sectoral_growth.json': sectoral_growth,
    'credit_sector_shares.json': sector_shares,
    'credit_summary.json': summary,
}
for fname, data in files.items():
    with open(OUT / fname, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'✅ {fname}')

print('\n🎉 Done! Run frontend to see the dashboard.')

✅ credit_monthly.json
✅ credit_sectoral.json
✅ credit_sectoral_growth.json
✅ credit_sector_shares.json
✅ credit_summary.json

🎉 Done! Run frontend to see the dashboard.
